## 👋 About me

Publishing this as a community hobby project. I'm not eligible for the prize
pool. Feel free to fork, modify, ... . if this saves someone a day
of debugging, it's done its job.

**I'm looking for work** 🔍 — this notebook is also my portfolio piece. If
you're hiring for Agentic AI engineering roles, please reach out:

📧 **re.azami@gmail.com**

Remote, hybrid, or on-site — all welcome. Happy to collaborate on projects too.

# 🇨🇭 Swiss Legal Retrieval — Corpus Indexing

Builds a searchable index of Swiss federal laws and court decision
considerations for the [LLM Agentic Legal Retrieval competition](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval).
Output: a `corpus.parquet` with about 2.9M chunks + bge-m3 embeddings
(173k from `laws_de.csv`, 2.74M from `court_considerations.csv`), plus all
dependencies the submission notebook needs to run offline.

**This is a utility notebook** — attach it to your submission notebook via
`Add Input`, install from its wheels, load the parquet. All the expensive work
(model download, 3M chunks encoded on GPU) happens once, here.

## 🎯 What it does
1. 📦 Stages offline deps (wheels, bge-m3, Weaviate binary)
2. 📂 Loads `laws_de.csv` (176k rows) + `court_considerations.csv` (2.5M rows)
3. 🧹 Drops parser junk and dispositif boilerplate
4. ✂️ Chunks long docs (different strategy per file)
5. ⚡ Encodes with bge-m3 in fp16 on dual T4
6. 💾 Writes `corpus.parquet`

## 💡 Key choices
- **bge-m3** — multilingual, handles EN↔DE queries, strong on MTEB
- **fp16** — 4.6× faster than fp32 on T4, no measurable quality loss
- **Weaviate** — native hybrid (BM25 + vector) search in one call
- **Dual chunking strategy** — paragraph-packed for laws, positional for decisions

## 📁 1. Setup

Create the directory layout. Everything under `/kaggle/working` at the end of
this notebook becomes the attachable artifact that downstream notebooks can
mount via `Add Input`, so these paths are effectively the public API:

- `wheels/` — offline-installable Python packages
- `models/bge-m3/` — embedding model weights
- `weaviate_binary/` — the Weaviate server executable
- `corpus.parquet` — the final indexed corpus (built later)

In [ ]:
# Cell 1 — directories
!mkdir -p /kaggle/working/wheels
!mkdir -p /kaggle/working/models
!mkdir -p /kaggle/working/weaviate_binary
!mkdir -p /kaggle/working/weaviate_data

## 📦 2. Pre-staging wheels for offline install

Kaggle submission notebooks run with **no internet**, so every package the
submission notebook will use has to already be on disk. `pip download` (note:
*download*, not *install*) accumulates wheels in `/kaggle/working/wheels/`
without touching this notebook's environment.

Later, the submission notebook runs:

pip install --no-index --find-links wheels/ weaviate-client sentence-transformers

which installs purely from these local files. Zero network calls.

Versions are pinned to the latest stable as of April 2026. Keeping them in sync
between this notebook and the submission notebook matters — weaviate-client
and the Weaviate server binary have tightly coupled version compatibility, and
mismatches fail silently in weird ways.

In [ ]:
# Cell 2 — wheels (current as of April 2026)
!pip download \
    --dest /kaggle/working/wheels \
    --python-version 3.12 \
    --only-binary=:all: \
    weaviate-client==4.20.5 \
    sentence-transformers==5.4.0 \
    transformers==5.5.3 \
    tokenizers \
    huggingface_hub \
    safetensors \
    accelerate \
    numpy \
    pandas \
    pyarrow \
    tqdm

## 🗃️ 3. Downloading the Weaviate server binary

Important thing: **Weaviate is a Go server, not a
Python library.** The `weaviate-client` package only wraps gRPC calls — it
can't store or search vectors by itself. To use Weaviate at all, there has to
be a running Weaviate server somewhere.

For a Kaggle-offline workflow, we download the Linux server binary from GitHub
releases here and ship it as part of this notebook's output. The submission
notebook then uses **Embedded Weaviate**, which spawns this binary as a
subprocess inside the Python process. No Docker, no cloud, no network.

Server **1.36.0** pairs with Python client **4.20.5** per Weaviate's
[compatibility table](https://docs.weaviate.io/weaviate/client-libraries/python).
This pairing matters — Weaviate's on-disk state format is version-sensitive,
and mixing versions between the server that writes data and the one that reads
it can fail with cryptic errors.

In [ ]:
# Cell 3 — Weaviate server binary (1.36.0 pairs with client 4.20.x per docs table)
import urllib.request, tarfile, os

WEAVIATE_VERSION = "1.36.0"
url = (f"https://github.com/weaviate/weaviate/releases/download/"
       f"v{WEAVIATE_VERSION}/weaviate-v{WEAVIATE_VERSION}-linux-amd64.tar.gz")
tar_path = "/kaggle/working/weaviate_binary/weaviate.tar.gz"

print(f"Downloading {url}")
urllib.request.urlretrieve(url, tar_path)
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall("/kaggle/working/weaviate_binary", filter="data")
os.remove(tar_path)
os.chmod("/kaggle/working/weaviate_binary/weaviate", 0o755)
print("Weaviate binary ready")

In [ ]:
!ls -lh /kaggle/working/weaviate_binary/
!/kaggle/working/weaviate_binary/weaviate --version

## 🤖 4. Downloading the embedding model

[**BAAI/bge-m3**](https://huggingface.co/BAAI/bge-m3) is our embedding model.
Three reasons:

1. **Cross-lingual retrieval.** The competition queries may be in English
   while the corpus is in German. bge-m3 was trained on 100+ languages and
   handles EN↔DE retrieval natively — no need to translate queries first.
2. **Long context.** Supports up to 8192 tokens, though we use 1024 in
   practice to bound encoding time.
3. **Strong benchmarks.** Top results on MTEB for multilingual retrieval
   compared to other open-source options.

Download is ~2.3 GB. We skip ONNX and OpenVINO variants because we'll run the
PyTorch weights on GPU and those other formats would just take disk space.

In [ ]:
# Cell 4 — download the embedding model (BGE-M3, ~2.3 GB)
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="BAAI/bge-m3",
    local_dir="/kaggle/working/models/bge-m3",
    local_dir_use_symlinks=False,
    ignore_patterns=["*.onnx", "*onnx*", "openvino*", "*.msgpack", "*.h5", "*.ot"],
)
print("bge-m3 ready")

## 🔧 5. Installing packages for this notebook's own use

The wheels in Section 2 were staged for the *submission* notebook. But this
utility notebook itself also needs `sentence-transformers` (to encode the
corpus) and `weaviate-client` (not actually used here, but installed for
compatibility testing).

Using `--no-index --find-links` forces pip to install from our local wheels
folder only. This guarantees this notebook and the submission notebook use
**identical versions** — no "works on my machine" surprises from Kaggle's
pre-installed packages drifting.

In [ ]:
# Cell 5 — install in this notebook from the local wheels
!pip install --quiet \
    --no-index --find-links /kaggle/working/wheels \
    weaviate-client==4.20.5 sentence-transformers==5.4.0

## 📂 6. Loading the raw corpus

Two CSV files from the competition dataset:

- **`laws_de.csv`** — 176k Swiss federal statute articles. Each row is one
  article or sub-article. Columns: `citation` (e.g. `Art. 963 Abs. 1 ZGB`),
  `text` (German legal prose), `title` (the full law's name).
- **`court_considerations.csv`** — 2.5M individual "considerations" extracted
  from Swiss federal court decisions. Each row is one numbered consideration
  from one ruling. Columns: `citation` (e.g. `BGE 139 I 2 E. 2.1`), `text`.

These get loaded into pandas DataFrames and then processed by the filtering
and chunking logic below.

In [ ]:
# Cell 6 — load the corpus CSVs
import pandas as pd

LAWS_PATH = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv"
DECISIONS_PATH = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv"


laws = pd.read_csv(LAWS_PATH)
decisions = pd.read_csv(DECISIONS_PATH)

print("laws_de.csv columns:", list(laws.columns))
print("laws_de.csv rows:", len(laws))
print("court_considerations.csv columns:", list(decisions.columns))
print("court_considerations.csv rows:", len(decisions))
print("\nlaws sample:\n", laws.head(2))
print("\ndecisions sample:\n", decisions.head(2))

## 🧹 7. Cleaning, filtering, chunking

This is the most important cell in the notebook. It does three things that
each deserve explanation, because the "right" thresholds here came from
looking at the actual data carefully, not from intuition.

### Filtering out noise

Both CSVs contain a lot of content that *looks* like real text but is actually
garbage that hurts retrieval. Concretely:

**In decisions**, about **198k rows (8%) are dispositif boilerplate** — the
formal one-line rulings Swiss courts append to every decision, repeated
thousands of times verbatim across unrelated cases. Examples:
- `"Die Beschwerde wird abgewiesen."` (appeal dismissed) — appears 15,912 times
- `"Auf die Beschwerde wird nicht eingetreten."` (appeal not entertained) — 28,634 times
- `"Le recours est rejeté."` — 10,869 times
- `"Il ricorso è respinto."` — 1,077 times

These have zero retrieval value. They're identical across completely unrelated
cases and would just clutter the index with noise. They all land under 50
characters, so **`DEC_MIN_CHARS = 50`** drops them cleanly.

**In laws**, the noise is different: about **4k rows (2%) are parser artifacts**
like `'…4'` or `'3 …15'` (upstream truncation markers), or the word
`'Aufgehoben'` ("repealed" — articles with no remaining text), or very short
cross-references like `'Zu Art. 52 Abs. 5'`. We drop anything under 15 chars,
or containing `'Aufgehoben'`, or containing the `…` truncation marker at
short length. **`LAWS_MIN_CHARS = 15`** with pattern filters.

The asymmetry matters: using the same threshold for both files would drop
legitimate short statute articles from laws (plenty of Swiss constitutional
articles are single sentences like `"Die Würde des Menschen ist unantastbar."`
at 38 chars) while failing to catch the boilerplate in decisions.

### Chunking long documents

bge-m3's practical context is 1024 tokens (~4000 chars). The longest court
decision in our corpus is 92,652 characters — a single row that would lose
most of its content to tokenizer truncation. We need to split long documents.

The surprising bit: **laws have clean paragraph structure (newlines separating
legal articles), but decisions have no paragraph breaks at all.** Zero.
Decisions are stored as single unbroken strings, even when the underlying
content is 20 pages of court reasoning. So we use two different strategies:

- **Laws** → greedy-pack paragraphs on `\n` boundaries into ~2000-char chunks.
  Laws are short enough that we almost never need to cut inside a paragraph.
- **Decisions** → positional sliding window: chunk at exactly 2000 chars with
  200 chars of overlap between adjacent chunks. Cuts mid-sentence sometimes,
  but bge-m3 is robust to that and it keeps the code simple.

A sentence-aware splitter would produce cleaner chunks for decisions, but
Swiss legal German has enough abbreviations (`Art.`, `Abs.`, `E.`, `BGE`, etc.)
that every period looks like a sentence boundary. The complexity isn't worth
the small quality gain.

### Output

The cell produces a single `records` list: ~2.9M dicts, each with fields
`citation, text, title, embed_text, source_type, chunk_id`. `embed_text` is
what gets passed to the model (for laws, it's `title + text` so the embedding
captures both context and content); the other fields are for storage and
retrieval.

In [ ]:
# Cell 7 — clean, filter, and chunk both files (complete version)
import gc, re

LAWS_MIN_CHARS = 15
DEC_MIN_CHARS  = 50
LAWS_JUNK_PATTERNS = ("aufgehoben",)

def is_truncation_artifact(txt):
    stripped = txt.strip()
    return "…" in stripped and len(stripped) < 20

def is_law_junk(txt):
    low = txt.lower().strip()
    if any(p in low for p in LAWS_JUNK_PATTERNS):
        return True
    if is_truncation_artifact(txt):
        return True
    return False

def clean(s):
    if s is None:
        return ""
    s = str(s).strip()
    return "" if s == "nan" else s


# ---- LAWS: greedy-pack newline paragraphs into ~size-char chunks ----
def chunk_laws(text, size=2000, min_chunk=200):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    if not paras:
        stripped = text.strip()
        return [stripped] if stripped else []
    
    chunks, buf, buf_len = [], [], 0
    for p in paras:
        if buf and buf_len + len(p) + 1 > size and buf_len >= min_chunk:
            chunks.append("\n".join(buf))
            buf, buf_len = [], 0
        buf.append(p)
        buf_len += len(p) + 1
    if buf:
        chunks.append("\n".join(buf))
    return chunks


# ---- DECISIONS: positional sliding window with overlap ----
def chunk_decisions(text, size=2000, overlap=200):
    text = text.strip()
    if len(text) <= size:
        return [text] if text else []
    
    chunks, step, start = [], size - overlap, 0
    n = len(text)
    while start < n:
        chunks.append(text[start:start + size])
        if start + size >= n:
            break
        start += step
    return chunks


# ---- Build records ----
records = []

# Laws
n_laws_kept = 0
for _, row in laws.iterrows():
    cit   = clean(row["citation"])
    txt   = clean(row["text"])
    title = clean(row["title"])
    if not cit or len(txt) < LAWS_MIN_CHARS or is_law_junk(txt):
        continue
    n_laws_kept += 1
    for i, ch in enumerate(chunk_laws(txt)):
        embed_text = f"{title}\n\n{ch}" if title else ch
        records.append({
            "citation":    cit,
            "text":        ch,
            "title":       title,
            "embed_text":  embed_text,
            "source_type": "law",
            "chunk_id":    i,
        })
n_laws_chunks = len(records)

# Decisions
n_dec_kept = 0
for _, row in decisions.iterrows():
    cit = clean(row["citation"])
    txt = clean(row["text"])
    if not cit or len(txt) < DEC_MIN_CHARS:
        continue
    n_dec_kept += 1
    for i, ch in enumerate(chunk_decisions(txt)):
        records.append({
            "citation":    cit,
            "text":        ch,
            "title":       "",
            "embed_text":  ch,
            "source_type": "decision",
            "chunk_id":    i,
        })

del laws, decisions
gc.collect()

n_dec_chunks = len(records) - n_laws_chunks
print(f"Laws:      {n_laws_kept:>10,} kept → {n_laws_chunks:>10,} chunks")
print(f"Decisions: {n_dec_kept:>10,} kept → {n_dec_chunks:>10,} chunks")
print(f"Total chunks: {len(records):,}")

## ⚡ 8. Encoding with fp16 on dual T4

The long cell. Encodes all ~2.9M chunks on two T4 GPUs in parallel. Several
important tricks here, because naive encoding would take far longer than
Kaggle's 12-hour session limit.

### The fp16 trick — 4.6× faster than fp32

T4 GPUs have dedicated fp16 tensor cores that run ~4× faster than fp32 compute
for matrix multiplications. bge-m3 inference is almost entirely matmuls, so
flipping to fp16 gives a massive speedup for essentially free.

The `embedder.half()` call converts the model weights from fp32 to fp16,
activating the tensor cores. Output embeddings come back as fp16 naturally.
For **L2-normalized retrieval vectors** (which is what we're producing, with
`normalize_embeddings=True`), the precision loss compared to fp32 is
indistinguishable from noise — the vectors all live in [-0.1, 0.1] range,
well within fp16's representable precision, and cosine similarity is robust
to small perturbations.

Without `.half()`, this cell would take over 24 hours on dual T4 and fail to
fit in one Kaggle session. With `.half()`, it takes ~5 hours.

### Dual-GPU with `encode_multi_process`

`SentenceTransformer.encode` only uses one GPU by default. `encode_multi_process`
spawns worker processes that each own one GPU, splits the input list, and
merges results. Roughly 1.7× additional speedup over single-GPU (not quite 2×
because of IPC overhead and imbalanced shards at the end).

### Shard checkpointing

Kaggle notebooks occasionally die — OOM, timeout, network hiccup, accidental
cell rerun. Losing 4 hours of encoding progress to a crash is painful and
avoidable.

The solution: save each 200k-chunk batch to disk as a separate `.npy` file as
soon as it's computed. If the notebook dies, rerun the cell — the loop checks
`shard_XXX.npy` existence before encoding and skips shards that already exist.
Worst-case loss from a crash: 1 shard (~15 minutes), not everything.

We store shards as fp16 on disk (half the size of fp32) to fit under Kaggle's
19.5 GB output limit, and delete them after the final `np.concatenate` inside
this cell — Cell 9's parquet write needs the freed disk space.

### Cleaning up before Cell 9

The first time I ran this pipeline, Cell 8 finished successfully but Cell 9
OOMed 73 seconds later. The problem: Cell 8 leaves ~10 GB of dead weight in
RAM when it finishes — the `texts` list (the input we passed to
`encode_multi_process`), the `embedder` model, and the worker process pool
references. None of them were needed by Cell 9, but Python can't garbage-collect
them until the names go out of scope.

The fix is explicit: `del texts` inside the `try` block, and `del embedder; del pool;
gc.collect(); torch.cuda.empty_cache()` in the `finally` block so it runs even
if encoding fails partway through. The `psutil` report at the very end of the
cell prints RAM usage to the save log, giving us an early warning if cleanup
didn't work before we commit to the 5-minute parquet write.

After cleanup, RAM usage should be roughly `records` (about 5 GB) + `embeddings`
(about 6 GB) + Python baseline (about 2 GB) ≈ 13 GB out of 30 GB. Plenty of headroom
for Cell 9.

In [ ]:
# Cell 8 — encode with fp16 throughout, store fp16 on disk, free memory before Cell 9
import torch, time, shutil, gc, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

assert torch.cuda.device_count() >= 2, "Set Accelerator → T4 x2 in Settings"

CHECKPOINT_DIR = Path("/kaggle/working/embedding_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
SHARD_SIZE = 200_000

texts = [r["embed_text"] for r in records]
total = len(texts)
n_shards = (total + SHARD_SIZE - 1) // SHARD_SIZE
print(f"Total chunks: {total:,}  →  {n_shards} shards of {SHARD_SIZE:,}")

embedder = SentenceTransformer("/kaggle/working/models/bge-m3")
embedder.max_seq_length = 1024
embedder.half()   # fp16 on GPU — both compute and output are fp16

pool = embedder.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])

try:
    all_shards = []
    t_start = time.time()
    
    for shard_idx, shard_start in enumerate(range(0, total, SHARD_SIZE)):
        shard_path = CHECKPOINT_DIR / f"shard_{shard_idx:03d}.npy"
        shard_end = min(shard_start + SHARD_SIZE, total)
        
        if shard_path.exists():
            print(f"[{shard_idx+1}/{n_shards}] Loading checkpoint: {shard_path.name}")
            shard_vecs = np.load(shard_path)  # loads as fp16, keeps fp16
        else:
            print(f"[{shard_idx+1}/{n_shards}] Encoding {shard_start:,}:{shard_end:,} ...")
            t0 = time.time()
            shard_vecs = embedder.encode_multi_process(
                texts[shard_start:shard_end],
                pool,
                batch_size=64,
                normalize_embeddings=True,
            ).astype(np.float16)   # keep fp16 — honest representation
            np.save(shard_path, shard_vecs)
            
            elapsed  = time.time() - t0
            total_el = time.time() - t_start
            done     = shard_end
            eta_min  = (total - done) * (total_el / done) / 60 if done > 0 else 0
            print(f"    done in {elapsed/60:.1f} min  "
                  f"| total: {total_el/60:.1f} min  "
                  f"| ETA: {eta_min:.1f} min")
        
        all_shards.append(shard_vecs)
    
    embeddings = np.concatenate(all_shards, axis=0)
    del all_shards
    
    print(f"\nFinal embeddings shape: {embeddings.shape}")
    print(f"dtype: {embeddings.dtype}")
    print(f"Size in memory: {embeddings.nbytes / 1e9:.2f} GB")
    print(f"Total encoding time: {(time.time() - t_start)/60:.1f} min")
    
    shutil.rmtree(CHECKPOINT_DIR)
    print("Removed shard checkpoints")
    
    # Free the input text list — no longer needed, ~3-4 GB
    del texts
    print("Freed texts list")

finally:
    # Shut down the multi-process pool cleanly
    embedder.stop_multi_process_pool(pool)
    
    # Drop references so Python can garbage-collect the model and workers
    del embedder
    del pool
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    print("Freed embedder, pool, CUDA cache")

# Report RAM state so Cell 9 knows what it's working with
import psutil
mem = psutil.virtual_memory()
print(f"\nRAM after Cell 8 cleanup: {mem.used/1e9:.1f} GB used / "
      f"{mem.total/1e9:.1f} GB total ({mem.percent:.0f}%)")

## 💾 9. Writing the final parquet — carefully

`corpus.parquet` is what downstream notebooks will actually consume. One row
per chunk, with these columns:

| Column | Type | Purpose |
|---|---|---|
| `citation` | string | The citation key (e.g. `Art. 963 Abs. 1 ZGB`) |
| `text` | string | The chunk's text content (for display / reranking) |
| `title` | string | The parent law's name (laws only; empty for decisions) |
| `source_type` | string | `"law"` or `"decision"` — for filtering at ingest |
| `chunk_id` | int | Which chunk of the parent doc (0 for un-chunked) |
| `vector` | float16[1024] | The bge-m3 embedding |

Vectors stored as **float16** (not float32) because the underlying precision
from fp16 encoding is already fp16 — converting to fp32 for storage would
just pad the bits with zeros and double the disk size for zero information
gain. Submission notebooks cast back to fp32 at ingest time (Weaviate requires
fp32 for its HNSW index).

Compressed with zstd. Final file size: around 6–7 GB.

### Why streaming the write instead of one big `to_parquet` call

The naive approach would be:

```python
df = pd.DataFrame([{k: v for k, v in r.items() if k != "embed_text"} for r in records])
df["vector"] = list(embeddings)
df.to_parquet(OUT, compression="zstd")
```

This doesn't fit in RAM. Three problems at once:

1. **The list comprehension materializes a second full copy of records** — during
   the `[{...} for r in records]` expression, Python holds both the original
   `records` list and the new list of copied dicts simultaneously. That's ~10 GB
   just for the records data.
2. **`list(embeddings)` creates a Python list of 2.9M numpy arrays**, adding
   another ~6 GB of object overhead on top of the existing `embeddings` ndarray.
3. **`df.to_parquet` buffers the entire DataFrame** before writing, so memory
   usage peaks near the total size of everything combined.

Peak memory for the naive version on this corpus: ~28 GB. Kaggle has 30 GB.
One bad allocation and you OOM — which is exactly what happened on our first
run. Five hours of encoding, gone.

The streaming approach avoids this entirely:

- Iterate through `records` and `embeddings` in 100k-row slices
- Build one small DataFrame per slice (~1 GB peak per batch)
- Convert to a pyarrow Table, write it through `ParquetWriter`, free it
- Force `gc.collect()` every few batches to keep temporary allocations from
  piling up

Peak memory during Cell 9 with this approach: ~13 GB, with 17 GB of headroom.
The `psutil` print at the start of the cell confirms the starting RAM state
in the save log — if it shows >20 GB used, Cell 8's cleanup didn't work and
we should abort before wasting time on a doomed write.

### The explicit pyarrow schema

Writing fp16 vector columns through pandas is finicky — pyarrow sometimes
silently converts `float16` arrays to `float32` during type inference, or
coerces fixed-size vector lists to variable-length lists. Passing an explicit
`pa.schema(...)` with `pa.list_(pa.float16(), 1024)` forces the exact types
we want and fails loudly if something doesn't match, instead of silently
corrupting the output.

In [ ]:
# Cell 9 — stream-write parquet in chunks, fp16 throughout
import gc, os
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Quick sanity on RAM state at the start of Cell 9
import psutil
mem = psutil.virtual_memory()
print(f"RAM at start of Cell 9: {mem.used/1e9:.1f} GB / {mem.total/1e9:.1f} GB ({mem.percent:.0f}%)")

OUT = "/kaggle/working/corpus.parquet"
CHUNK = 100_000

schema = pa.schema([
    ("citation",    pa.string()),
    ("text",        pa.string()),
    ("title",       pa.string()),
    ("source_type", pa.string()),
    ("chunk_id",    pa.int32()),
    ("vector",      pa.list_(pa.float16(), 1024)),
])

total = len(records)
print(f"\nStreaming {total:,} rows to {OUT} in batches of {CHUNK:,}")
print(f"Batches: {(total + CHUNK - 1) // CHUNK}")

with pq.ParquetWriter(OUT, schema, compression="zstd") as writer:
    for start in range(0, total, CHUNK):
        end = min(start + CHUNK, total)
        
        batch_df = pd.DataFrame({
            "citation":    [records[i]["citation"]    for i in range(start, end)],
            "text":        [records[i]["text"]        for i in range(start, end)],
            "title":       [records[i]["title"]       for i in range(start, end)],
            "source_type": [records[i]["source_type"] for i in range(start, end)],
            "chunk_id":    [records[i]["chunk_id"]    for i in range(start, end)],
            "vector":      list(embeddings[start:end]),
        })
        
        table = pa.Table.from_pandas(batch_df, schema=schema, preserve_index=False)
        writer.write_table(table)
        
        del batch_df, table
        if (start // CHUNK) % 5 == 0:
            gc.collect()
            print(f"  {end:,} / {total:,}")

size_gb = os.path.getsize(OUT) / 1e9
print(f"\nWrote {OUT}: {size_gb:.2f} GB, {total:,} rows")

# Final RAM report
mem = psutil.virtual_memory()
print(f"RAM after write: {mem.used/1e9:.1f} GB / {mem.total/1e9:.1f} GB ({mem.percent:.0f}%)")

## ✅ 10. Verifying the output directory

Sanity check: does `/kaggle/working` contain everything a downstream notebook
needs, and does it fit under Kaggle's 19.5 GB output limit?

Expected contents:
- `wheels/` (~2.7 GB)
- `models/bge-m3/` (~2.3 GB)
- `weaviate_binary/` (~140 MB)
- `corpus.parquet` (~6–7 GB)

Total should be around 12–13 GB — well under the limit, with comfortable
margin for any future additions.

In [ ]:
# Cell 10 — verify output directory structure
import os

print("Final /kaggle/working contents:")
for entry in sorted(os.listdir("/kaggle/working")):
    path = os.path.join("/kaggle/working", entry)
    if os.path.isdir(path):
        size = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, _, fs in os.walk(path) for f in fs)
        print(f"  {entry}/  ({size/1e9:.2f} GB)")
    else:
        print(f"  {entry}  ({os.path.getsize(path)/1e9:.2f} GB)")

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk("/kaggle/working") for f in fs
)
print(f"\nTotal output: {total/1e9:.2f} GB / 19.5 GB limit")